# CS4100 Final Project  
Team Members: Khushi Khan, Dustin Zhang, Kayla Handley, Koena Gupta

In [1159]:
import pandas as pd
import re
from pandas.api.types import is_numeric_dtype, is_string_dtype

In [1160]:
# Load datasets
df_low = pd.read_csv('../data/raw/low_popularity_spotify_data.csv', low_memory=False)
df_high = pd.read_csv('../data/raw/high_popularity_spotify_data.csv', low_memory=False)
df_apr = pd.read_csv('../data/raw/SpotifyAudioFeaturesApril2019.csv', low_memory=False)
df_nov = pd.read_csv('../data/raw/SpotifyAudioFeaturesNov2018.csv', low_memory=False)

all_datasets = {'Low Popularity Songs': df_low, 
                'High Popularity Songs': df_high, 
                'Spotify Features April 2019': df_apr, 
                'Spotify Features Nov 2018': df_nov}

print("Dataset Shapes:")
for df_title, df in all_datasets.items():
    print(f"{df_title}: {df.shape}")

Dataset Shapes:
Low Popularity Songs: (3145, 29)
High Popularity Songs: (1686, 29)
Spotify Features April 2019: (130663, 17)
Spotify Features Nov 2018: (116372, 17)


In [1161]:
# Converting the column names to lowercase
for df_title, df in all_datasets.items():
    df.columns = df.columns.str.strip().str.lower()
    print(f"{df_title}: {df.columns}")

Low Popularity Songs: Index(['time_signature', 'track_popularity', 'speechiness', 'danceability',
       'playlist_name', 'track_artist', 'duration_ms', 'energy',
       'playlist_genre', 'playlist_subgenre', 'track_href', 'track_name',
       'mode', 'uri', 'type', 'track_album_release_date', 'analysis_url', 'id',
       'instrumentalness', 'track_album_id', 'playlist_id', 'track_id',
       'valence', 'key', 'tempo', 'loudness', 'acousticness', 'liveness',
       'track_album_name'],
      dtype='object')
High Popularity Songs: Index(['energy', 'tempo', 'danceability', 'playlist_genre', 'loudness',
       'liveness', 'valence', 'track_artist', 'time_signature', 'speechiness',
       'track_popularity', 'track_href', 'uri', 'track_album_name',
       'playlist_name', 'analysis_url', 'track_id', 'track_name',
       'track_album_release_date', 'instrumentalness', 'track_album_id',
       'mode', 'key', 'duration_ms', 'acousticness', 'id', 'playlist_subgenre',
       'type', 'playlist_i

In [1162]:
# Dropping unnamed columns
for df_title, df in all_datasets.items():
    print(df_title + " Dataset")
    print(f"Column count before dropping unnamed columns: {len(df.columns)}")
    df.drop(columns=df.columns[df.columns.str.contains("^unnamed")], inplace=True)
    print(f"Column count after dropping unnamed columns: {len(df.columns)}\n")

Low Popularity Songs Dataset
Column count before dropping unnamed columns: 29
Column count after dropping unnamed columns: 29

High Popularity Songs Dataset
Column count before dropping unnamed columns: 29
Column count after dropping unnamed columns: 29

Spotify Features April 2019 Dataset
Column count before dropping unnamed columns: 17
Column count after dropping unnamed columns: 17

Spotify Features Nov 2018 Dataset
Column count before dropping unnamed columns: 17
Column count after dropping unnamed columns: 17



In [1163]:
# Finding the columns with object types
for df_title, df in all_datasets.items():
    object_cols = df.select_dtypes(include='object').columns
    print(f"Columns in {df_title} Dataset:\n{object_cols}\n")

Columns in Low Popularity Songs Dataset:
Index(['playlist_name', 'track_artist', 'playlist_genre', 'playlist_subgenre',
       'track_href', 'track_name', 'uri', 'type', 'track_album_release_date',
       'analysis_url', 'id', 'track_album_id', 'playlist_id', 'track_id',
       'track_album_name'],
      dtype='object')

Columns in High Popularity Songs Dataset:
Index(['playlist_genre', 'track_artist', 'track_href', 'uri',
       'track_album_name', 'playlist_name', 'analysis_url', 'track_id',
       'track_name', 'track_album_release_date', 'track_album_id', 'id',
       'playlist_subgenre', 'type', 'playlist_id'],
      dtype='object')

Columns in Spotify Features April 2019 Dataset:
Index(['artist_name', 'track_id', 'track_name'], dtype='object')

Columns in Spotify Features Nov 2018 Dataset:
Index(['artist_name', 'track_id', 'track_name'], dtype='object')



In [1164]:
common_cols = list(set(df_apr.columns) & set(df_high.columns) & set(df_low.columns) & set(df_nov.columns))
print(f"Common columns across all datasets:\n{common_cols}")

Common columns across all datasets:
['danceability', 'liveness', 'energy', 'track_name', 'key', 'duration_ms', 'instrumentalness', 'valence', 'acousticness', 'loudness', 'track_id', 'time_signature', 'speechiness', 'mode', 'tempo']


Since we'll be combining these datasets later on, we'll focus on the object-type columns `track_id`, `artist_name` and `track_name`. We'll index each dataset using `track_id` since it's a common column across all datasets and likely to be unique as all of these datasets were created using Spotify data. We'll retain `artist_name` and `track_name` as they contain valuable information for the user once our model outputs a playlist. 

In [1165]:
# Updating types for the `artist_name` and `track_name` columns to string
for df_title, df in all_datasets.items():
    if "track_name" in df.columns:
        df["track_name"] = df["track_name"].astype(str).str.strip().str.lower()
    if "artist_name" in df.columns:
        df["artist_name"] = df["artist_name"].astype(str).str.strip().str.lower()

In [1166]:
# Adding a column to each dataset to reference its original source
df_low["source"] = "low_popularity"
df_high["source"] = "high_popularity"
df_apr["source"] = "april_2019"
df_nov["source"] = "nov_2018"

In [1167]:
# Concatenating all the datasets
combined_df = pd.concat(
    [df_high, df_low, df_nov, df_apr], 
    axis=0, 
    ignore_index=True, 
    sort=False
)

# Ensuring that the datasets are groupyed by their `track_id``
combined_df = combined_df.groupby('track_id', as_index=False).first()

# Viewing the resulting dimensions of the dataset
print(combined_df.shape)

(135419, 32)


In [1168]:
# Checking the distribution of the data's sources
print(combined_df["source"].value_counts())

source
nov_2018           116145
april_2019          14779
low_popularity       3058
high_popularity      1437
Name: count, dtype: int64


In [1169]:
# Viewing how the combined dataset looks
combined_df.head()

,track_id,energy,tempo,danceability,playlist_genre,loudness,liveness,valence,track_artist,time_signature,...,key,duration_ms,acousticness,id,playlist_subgenre,type,playlist_id,source,artist_name,popularity
0,0009UBVA8DCDwk1Hepib6P,0.395,124.018,0.680,None,-13.803,0.0958,0.0938,None,4.0,...,10.0,188813.0,0.074200,None,None,None,None,nov_2018,pussygangco,0.0
1,000RDCYioLteXcutOjeweY,0.770,161.721,0.679,None,-3.537,0.0825,0.8390,None,4.0,...,0.0,190203.0,0.058300,None,None,None,None,nov_2018,jordan sandhu,48.0
2,000TqGTOAZjAIFU6SjmFmc,0.898,137.949,0.399,None,-10.859,0.2190,0.0529,None,4.0,...,4.0,336100.0,0.000036,None,None,None,None,april_2019,ram,23.0
3,000v2QpqP2NmikGmSkLlCZ,0.784,126.931,0.850,None,-6.794,0.1760,0.8440,None,3.0,...,5.0,134514.0,0.426000,None,None,None,None,nov_2018,panchito arredondo,18.0
4,002QT7AS6h1LAF5dla8D92,0.653,123.032,0.830,None,-5.298,0.1120,0.2280,None,4.0,...,1.0,207827.0,0.046900,None,None,None,None,nov_2018,young dolph,53.0


In [1170]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135419 entries, 0 to 135418
Data columns (total 32 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   track_id                  135419 non-null  object 
 1   energy                    135418 non-null  float64
 2   tempo                     135418 non-null  float64
 3   danceability              135418 non-null  float64
 4   playlist_genre            4495 non-null    object 
 5   loudness                  135418 non-null  float64
 6   liveness                  135418 non-null  float64
 7   valence                   135418 non-null  float64
 8   track_artist              4495 non-null    object 
 9   time_signature            135418 non-null  float64
 10  speechiness               135418 non-null  float64
 11  track_popularity          4495 non-null    float64
 12  track_href                4494 non-null    object 
 13  uri                       4494 non-null    o

In [1171]:
combined_df.isnull().sum()

track_id                         0
energy                           1
tempo                            1
danceability                     1
playlist_genre              130924
loudness                         1
liveness                         1
valence                          1
track_artist                130924
time_signature                   1
speechiness                      1
track_popularity            130924
track_href                  130925
uri                         130925
track_album_name            130925
playlist_name               130924
analysis_url                130925
track_name                       0
track_album_release_date    130924
instrumentalness                 1
track_album_id              130924
mode                             1
key                              1
duration_ms                      1
acousticness                     1
id                          130925
playlist_subgenre           130924
type                        130925
playlist_id         

In [1172]:
# Dropping columns if more than 60% of the column consists of null values
print(f"Number of columns before drop: {len(combined_df.columns)}")

threshold = 0.6

combined_df = combined_df.drop(columns=[
        col 
        for col in combined_df.columns 
        if combined_df[col].isnull().sum() / len(combined_df) > threshold
    ]
)

print(f"Number of columns after drop: {len(combined_df.columns)}")

Number of columns before drop: 32
Number of columns after drop: 18


In [1173]:
# Checking that little to no null values remain
combined_df.isnull().sum()

track_id               0
energy                 1
tempo                  1
danceability           1
loudness               1
liveness               1
valence                1
time_signature         1
speechiness            1
track_name             0
instrumentalness       1
mode                   1
key                    1
duration_ms            1
acousticness           1
source                 0
artist_name         4430
popularity          4430
dtype: int64

In [1174]:
# Checking the column types
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135419 entries, 0 to 135418
Data columns (total 18 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   track_id          135419 non-null  object 
 1   energy            135418 non-null  float64
 2   tempo             135418 non-null  float64
 3   danceability      135418 non-null  float64
 4   loudness          135418 non-null  float64
 5   liveness          135418 non-null  float64
 6   valence           135418 non-null  float64
 7   time_signature    135418 non-null  float64
 8   speechiness       135418 non-null  float64
 9   track_name        135419 non-null  object 
 10  instrumentalness  135418 non-null  float64
 11  mode              135418 non-null  float64
 12  key               135418 non-null  float64
 13  duration_ms       135418 non-null  float64
 14  acousticness      135418 non-null  float64
 15  source            135419 non-null  object 
 16  artist_name       13

In [1175]:
# Correcting the object type columns with string types
combined_df = combined_df.astype({
    'track_id': 'string',
    'track_name': 'string',
    'source': 'string',
    'artist_name': 'string',
})

The columns were correctly converted to the proper types.

In [1176]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135419 entries, 0 to 135418
Data columns (total 18 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   track_id          135419 non-null  string 
 1   energy            135418 non-null  float64
 2   tempo             135418 non-null  float64
 3   danceability      135418 non-null  float64
 4   loudness          135418 non-null  float64
 5   liveness          135418 non-null  float64
 6   valence           135418 non-null  float64
 7   time_signature    135418 non-null  float64
 8   speechiness       135418 non-null  float64
 9   track_name        135419 non-null  string 
 10  instrumentalness  135418 non-null  float64
 11  mode              135418 non-null  float64
 12  key               135418 non-null  float64
 13  duration_ms       135418 non-null  float64
 14  acousticness      135418 non-null  float64
 15  source            135419 non-null  string 
 16  artist_name       13

In [1177]:
# Checking the statistical properties of the unscaled data
combined_df.describe()

,energy,tempo,danceability,loudness,liveness,valence,time_signature,speechiness,instrumentalness,mode,key,duration_ms,acousticness,popularity
count,135418.000000,135418.000000,135418.000000,135418.000000,135418.000000,135418.000000,135418.000000,135418.000000,135418.000000,135418.000000,135418.000000,1.354180e+05,135418.000000,130989.000000
mean,0.569585,119.429109,0.582723,-9.956872,0.194040,0.440993,3.880754,0.111576,0.223559,0.606559,5.232561,2.124397e+05,0.342767,25.455115
std,0.260061,30.108670,0.190127,6.573104,0.166643,0.259201,0.511608,0.123603,0.360284,0.488515,3.601971,1.220101e+05,0.345131,18.992494
min,0.000000,0.000000,0.000000,-60.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.203000e+03,0.000000,0.000000
25%,0.397000,96.015250,0.461000,-11.849000,0.097400,0.225000,4.000000,0.038900,0.000000,0.000000,2.000000,1.637138e+05,0.032200,10.000000
50%,0.604000,120.021500,0.607000,-7.950000,0.123000,0.422000,4.000000,0.055800,0.000145,1.000000,5.000000,2.016530e+05,0.204000,23.000000
75%,0.775000,139.504000,0.728000,-5.668000,0.234000,0.640000,4.000000,0.128000,0.436000,1.000000,8.000000,2.407448e+05,0.636000,38.000000
max,1.000000,249.983000,0.996000,1.806000,0.999000,1.000000,5.000000,0.966000,1.000000,1.000000,11.000000,5.610020e+06,0.996000,100.000000


In [1178]:
# Min-Max scaling each column to ensure that column values remain between 0 and 1, reducing feature dominance
for col in combined_df.columns:
    if is_numeric_dtype(combined_df[col]):
        min_val = combined_df[col].min()
        max_val = combined_df[col].max()
        combined_df[col] = (combined_df[col] - min_val) / (max_val - min_val)

In [1179]:
# Verifying that columns were Min-Max scaled
combined_df.describe()

,energy,tempo,danceability,loudness,liveness,valence,time_signature,speechiness,instrumentalness,mode,key,duration_ms,acousticness,popularity
count,135418.000000,135418.000000,135418.000000,135418.000000,135418.000000,135418.000000,135418.000000,135418.000000,135418.000000,135418.000000,135418.000000,135418.000000,135418.000000,130989.000000
mean,0.569585,0.477749,0.585063,0.809681,0.194234,0.440993,0.776151,0.115503,0.223559,0.606559,0.475687,0.037318,0.344143,0.254551
std,0.260061,0.120443,0.190890,0.106351,0.166810,0.259201,0.102322,0.127953,0.360284,0.488515,0.327452,0.021761,0.346517,0.189925
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.397000,0.384087,0.462851,0.779067,0.097497,0.225000,0.800000,0.040269,0.000000,0.000000,0.181818,0.028628,0.032329,0.100000
50%,0.604000,0.480119,0.609438,0.842151,0.123123,0.422000,0.800000,0.057764,0.000145,1.000000,0.454545,0.035394,0.204819,0.230000
75%,0.775000,0.558054,0.730924,0.879073,0.234234,0.640000,0.800000,0.132505,0.436000,1.000000,0.727273,0.042367,0.638554,0.380000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


Each column has a minimum and maximum of 0 and 1 respectively, verifying that Min-Max scaling was properly performed.

In [1180]:
# Imputing data where data is null
for col in combined_df.columns:
    if is_numeric_dtype(combined_df[col]):
        # Fill null numeric columns with the column median
        combined_df[col] = combined_df[col].fillna(combined_df[col].median())
    elif is_string_dtype(combined_df[col]):
        # Fill null string type columns with an empty string
        combined_df[col] = combined_df[col].fillna("")

In [1181]:
combined_df.isnull().sum()

track_id            0
energy              0
tempo               0
danceability        0
loudness            0
liveness            0
valence             0
time_signature      0
speechiness         0
track_name          0
instrumentalness    0
mode                0
key                 0
duration_ms         0
acousticness        0
source              0
artist_name         0
popularity          0
dtype: int64

All null values were taken care of.

In [1182]:
# Dropping duplicate rows
print(f"Before dropping duplicates: {combined_df.shape}")

# Dropping rows with the same track name and artist name
combined_df = combined_df.drop_duplicates(subset=['track_name', 'artist_name']) 

print(f"After dropping duplicates: {combined_df.shape}")

Before dropping duplicates: (135419, 18)
After dropping duplicates: (134441, 18)


In [1183]:
# Dropping track id and grouping by `track_name` since we know it's unique now
combined_df = combined_df.groupby('track_name', as_index=False).first()
combined_df = combined_df.drop(columns=['track_id'])

In [1184]:
# Verifying that `track_id` was dropped and that the dataset is grouped by `track_name`
combined_df.head()

,track_name,energy,tempo,danceability,loudness,liveness,valence,time_signature,speechiness,instrumentalness,mode,key,duration_ms,acousticness,source,artist_name,popularity
0,!!!!,0.258,0.440074,0.758032,0.744005,0.109109,0.0593,0.8,0.087785,0.890,0.0,1.000000,0.024918,0.515060,nov_2018,alicks,0.24
1,!!!!!!!,0.278,0.000000,0.000000,0.620814,0.669670,0.0000,0.0,0.000000,0.000,1.0,0.090909,0.001850,0.771084,april_2019,billie eilish,0.33
2,"""1955",0.437,0.328034,0.317269,0.776171,0.090591,0.0393,0.6,0.037371,0.901,1.0,0.818182,0.033294,0.632530,nov_2018,pablo lópez,0.00
3,"""42"" - from sr3mm",0.563,0.520191,0.971888,0.861664,0.108108,0.3240,0.8,0.129400,0.000,1.0,0.090909,0.041881,0.002761,nov_2018,rae sremmurd,0.55
4,"""99""",0.804,0.383946,0.554217,0.901223,0.111111,0.7140,0.8,0.031366,0.000,1.0,0.727273,0.034995,0.006004,nov_2018,barns courtney,0.64


In [1185]:
# Removing punctuation and making cleaned text columns
def keep_alphanum(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

combined_df['track_name_clean'] = combined_df['track_name'].apply(keep_alphanum)
combined_df['artist_name_clean'] = combined_df['artist_name'].apply(keep_alphanum)

In [1186]:
combined_df.head()

,track_name,energy,tempo,danceability,loudness,liveness,valence,time_signature,speechiness,instrumentalness,mode,key,duration_ms,acousticness,source,artist_name,popularity,track_name_clean,artist_name_clean
0,!!!!,0.258,0.440074,0.758032,0.744005,0.109109,0.0593,0.8,0.087785,0.890,0.0,1.000000,0.024918,0.515060,nov_2018,alicks,0.24,,alicks
1,!!!!!!!,0.278,0.000000,0.000000,0.620814,0.669670,0.0000,0.0,0.000000,0.000,1.0,0.090909,0.001850,0.771084,april_2019,billie eilish,0.33,,billie eilish
2,"""1955",0.437,0.328034,0.317269,0.776171,0.090591,0.0393,0.6,0.037371,0.901,1.0,0.818182,0.033294,0.632530,nov_2018,pablo lópez,0.00,1955,pablo lópez
3,"""42"" - from sr3mm",0.563,0.520191,0.971888,0.861664,0.108108,0.3240,0.8,0.129400,0.000,1.0,0.090909,0.041881,0.002761,nov_2018,rae sremmurd,0.55,42 from sr3mm,rae sremmurd
4,"""99""",0.804,0.383946,0.554217,0.901223,0.111111,0.7140,0.8,0.031366,0.000,1.0,0.727273,0.034995,0.006004,nov_2018,barns courtney,0.64,99,barns courtney


In [1187]:
# Writing processed dataset to a CSV file
combined_df.to_csv("../data/cleaned/spotify_master_clean.csv", index=False)